# Azure Analytics — Synapse Serverless SQL, ADF, Event Hubs

## Mental Model

This notebook shows the **Azure analytics triangle**:

- **Synapse Serverless SQL** = query files in the lake without provisioning SQL infrastructure
- **Azure Data Factory (ADF)** = managed orchestration and copy pipelines
- **Event Hubs** = high-throughput event ingestion and fan-out consumption

**Citi narrative:** 6,000+ API endpoints are monitored for latency, error rate, and throughput. Alerts escalate through severity tiers. We will export endpoint + alert data from PostgreSQL to Parquet, show the Synapse Serverless SQL pattern for querying that lake data, define an ADF copy pipeline, and demonstrate Event Hubs producer/consumer behavior with isolated consumer groups.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import uuid
from pathlib import Path
from typing import Iterable, Optional

AZ_CMD = r"C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"
SUBSCRIPTION_ID = "b3811436-61fc-4a3a-a6a9-deb05955076d"

PG_HOST = "localhost"
PG_PORT = 5432
PG_DB = "de_telemetry"
PG_USER = "de_admin"
PG_PASSWORD = "DeAdmin2026!"

KAFKA_BOOTSTRAP = "localhost:9092"
KAFKA_CONTAINER = "citi_kafka"

print("Python:", sys.version.split()[0])
print("AZ CLI configured path:", AZ_CMD)
print("Azure subscription:", SUBSCRIPTION_ID)
print("Postgres target:", f"{PG_HOST}:{PG_PORT}/{PG_DB}")

def run_az(args: list[str], check: bool = True) -> subprocess.CompletedProcess:
    """
    Run az CLI using UTF-8, return CompletedProcess, and optionally tolerate missing CLI.
    """
    cmd = [AZ_CMD, *args]
    if not Path(AZ_CMD).exists():
        msg = f"Azure CLI not found at configured path: {AZ_CMD}"
        if check:
            raise FileNotFoundError(msg)
        print(msg)
        return subprocess.CompletedProcess(cmd, returncode=127, stdout="", stderr=msg)

    completed = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if check and completed.returncode != 0:
        raise RuntimeError(
            "az command failed\n"
            f"CMD: {' '.join(cmd)}\n"
            f"STDOUT:\n{completed.stdout}\n"
            f"STDERR:\n{completed.stderr}"
        )
    return completed

def safe_json_loads(text: str):
    text = (text or "").strip()
    if not text:
        return None
    return json.loads(text)

def maybe_import(name: str):
    try:
        module = __import__(name)
        print(f"Imported: {name}")
        return module
    except Exception as exc:
        print(f"Import unavailable: {name} -> {exc}")
        return None

psycopg2 = maybe_import("psycopg2")
pandas = maybe_import("pandas")
pyspark = maybe_import("pyspark")

try:
    from azure.eventhub import EventHubConsumerClient, EventData, EventHubProducerClient
    print("Imported: azure.eventhub")
except Exception as exc:
    EventHubConsumerClient = EventData = EventHubProducerClient = None
    print(f"Import unavailable: azure.eventhub -> {exc}")

try:
    from azure.storage.filedatalake import DataLakeServiceClient
    print("Imported: azure.storage.filedatalake")
except Exception as exc:
    DataLakeServiceClient = None
    print(f"Import unavailable: azure.storage.filedatalake -> {exc}")

## 1) Setup

We use a hardcoded Azure CLI path and the real subscription ID from the environment context.  
Packages are assumed preinstalled, so there are **no `%pip install`** cells.

In [ ]:
from datetime import datetime

session_suffix = uuid.uuid4().hex[:8]
location = "eastus"

resource_group = f"rg-synapse-adf-eh-{session_suffix}"
storage_account = f"delsa{session_suffix}"[:24]  # Azure storage acct rules
file_system_name = "telemetry"
local_export_dir = Path.cwd() / "artifacts" / f"azure_synapse_{session_suffix}"
local_export_dir.mkdir(parents=True, exist_ok=True)

eventhub_namespace = f"ehns-{session_suffix}"
eventhub_name = "telemetry-events"
consumer_group_a = "cg-investigations"
consumer_group_b = "cg-analytics"

print("resource_group =", resource_group)
print("storage_account =", storage_account)
print("file_system_name =", file_system_name)
print("local_export_dir =", local_export_dir)
print("eventhub_namespace =", eventhub_namespace)

## 2) Export endpoints + alerts to Parquet from PostgreSQL

We intentionally keep the dataset context fixed:

- `endpoints`: 10,000 rows
- `metrics`: 500,000 rows
- `alerts`: 25,000 rows

For the lake export we use **endpoints** and **alerts** only, matching the requested scenario.

In [ ]:
def postgres_connect():
    if psycopg2 is None:
        raise RuntimeError("psycopg2 is required but not available in this kernel.")
    return psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PASSWORD,
    )

def export_tables_to_parquet(export_dir: Path) -> dict[str, str]:
    """
    Export endpoints and alerts from Postgres to Parquet.
    Prefer Spark because it is explicitly available in the environment context.
    """
    export_dir.mkdir(parents=True, exist_ok=True)

    jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"
    jdbc_props = {
        "user": PG_USER,
        "password": PG_PASSWORD,
        "driver": "org.postgresql.Driver",
    }

    if pyspark is None:
        raise RuntimeError("pyspark is required but not available in this kernel.")

    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder
        .appName("azure-synapse-adf-eventhubs")
        .master("local[*]")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )

    outputs = {}
    for table_name in ["endpoints", "alerts"]:
        out_path = export_dir / table_name
        (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", table_name)
            .option("user", PG_USER)
            .option("password", PG_PASSWORD)
            .option("driver", "org.postgresql.Driver")
            .load()
            .write.mode("overwrite")
            .parquet(str(out_path))
        )
        outputs[table_name] = str(out_path)

    spark.stop()
    return outputs

parquet_outputs = None
try:
    parquet_outputs = export_tables_to_parquet(local_export_dir)
    print(json.dumps(parquet_outputs, indent=2))
except Exception as exc:
    print("Parquet export did not complete in this runtime.")
    print(type(exc).__name__, "->", exc)

## 3) ADLS Gen2 setup and upload pattern

This section uses Azure CLI to:

1. create a resource group  
2. create a storage account with **Hierarchical Namespace enabled**  
3. create a filesystem/container  
4. upload Parquet files

The code is written to be executable in a properly authenticated local setup.  
If the current notebook kernel is not logged into Azure, the cell reports that cleanly.

In [ ]:
def create_adls_gen2_resources() -> dict[str, str] | None:
    # Make sure CLI exists and subscription is selected
    run_az(["account", "set", "--subscription", SUBSCRIPTION_ID], check=True)

    run_az(["group", "create", "--name", resource_group, "--location", location], check=True)

    run_az([
        "storage", "account", "create",
        "--name", storage_account,
        "--resource-group", resource_group,
        "--location", location,
        "--sku", "Standard_LRS",
        "--kind", "StorageV2",
        "--hierarchical-namespace", "true",
    ], check=True)

    keys_cp = run_az([
        "storage", "account", "keys", "list",
        "--resource-group", resource_group,
        "--account-name", storage_account,
        "--output", "json",
    ], check=True)
    keys = safe_json_loads(keys_cp.stdout) or []
    account_key = keys[0]["value"]

    run_az([
        "storage", "fs", "create",
        "--name", file_system_name,
        "--account-name", storage_account,
        "--account-key", account_key,
    ], check=True)

    for name, folder_path in (parquet_outputs or {}).items():
        run_az([
            "storage", "fs", "directory", "upload",
            "--account-name", storage_account,
            "--account-key", account_key,
            "--file-system", file_system_name,
            "--source", folder_path,
            "--path", f"lake/{name}",
            "--recursive",
        ], check=True)

    return {
        "account_name": storage_account,
        "account_key": account_key,
        "file_system": file_system_name,
    }

adls_context = None
try:
    adls_context = create_adls_gen2_resources()
    print("Data uploaded to ADLS Gen2")
    print(json.dumps({k: v if k != "account_key" else "***redacted in notebook output***"
                      for k, v in adls_context.items()}, indent=2))
except Exception as exc:
    print("ADLS Gen2 setup/upload did not complete in this runtime.")
    print(type(exc).__name__, "->", exc)

## 4) Synapse Serverless SQL pattern

**Mental shortcut:** Synapse Serverless SQL is **Athena for Azure** — query files directly and pay per scan.

In practice, the SQL below is what you would run in **Synapse Studio** after the workspace is provisioned.

### Query Parquet directly with `OPENROWSET`

```sql
SELECT TOP 20 *
FROM OPENROWSET(
    BULK 'https://<storage-account>.dfs.core.windows.net/telemetry/lake/endpoints/',
    FORMAT = 'PARQUET'
) AS rows;
```

### External data source + file format + external table pattern

```sql
CREATE DATABASE SCOPED CREDENTIAL adls_cred
WITH IDENTITY = 'Managed Identity';

CREATE EXTERNAL DATA SOURCE telemetry_lake
WITH (
    LOCATION = 'abfss://telemetry@<storage-account>.dfs.core.windows.net',
    CREDENTIAL = adls_cred
);

CREATE EXTERNAL FILE FORMAT parquet_ff
WITH ( FORMAT_TYPE = PARQUET );

CREATE EXTERNAL TABLE dbo.endpoints_external
(
    endpoint_id INT,
    name VARCHAR(255),
    region VARCHAR(100),
    status VARCHAR(100),
    category VARCHAR(100)
)
WITH (
    LOCATION = '/lake/endpoints',
    DATA_SOURCE = telemetry_lake,
    FILE_FORMAT = parquet_ff
);

SELECT region, status, COUNT(*) AS endpoint_count
FROM dbo.endpoints_external
GROUP BY region, status
ORDER BY endpoint_count DESC;
```

In [ ]:
def build_synapse_sql(storage_account_name: str) -> str:
    return f"""
-- Query Parquet directly
SELECT TOP 20 *
FROM OPENROWSET(
    BULK 'https://{storage_account_name}.dfs.core.windows.net/{file_system_name}/lake/endpoints/',
    FORMAT = 'PARQUET'
) AS rows;

-- External data source pattern
CREATE DATABASE SCOPED CREDENTIAL adls_cred
WITH IDENTITY = 'Managed Identity';

CREATE EXTERNAL DATA SOURCE telemetry_lake
WITH (
    LOCATION = 'abfss://{file_system_name}@{storage_account_name}.dfs.core.windows.net',
    CREDENTIAL = adls_cred
);

CREATE EXTERNAL FILE FORMAT parquet_ff
WITH ( FORMAT_TYPE = PARQUET );

CREATE EXTERNAL TABLE dbo.alerts_external
(
    alert_id INT,
    endpoint_id INT,
    severity VARCHAR(50),
    message VARCHAR(8000),
    created_at DATETIME2
)
WITH (
    LOCATION = '/lake/alerts',
    DATA_SOURCE = telemetry_lake,
    FILE_FORMAT = parquet_ff
);
""".strip()

if adls_context:
    synapse_sql = build_synapse_sql(adls_context["account_name"])
    print(synapse_sql)
else:
    print("Synapse SQL pattern ready; execute after ADLS upload completes and Synapse workspace is provisioned.")

## 5) Azure Data Factory concepts

ADF pipelines are the classic **source → sink** orchestration pattern.

### Core ideas

- **Pipeline**: orchestration container
- **Activity**: one task inside a pipeline
- **Linked Service**: connection definition to a system
- **Dataset**: data shape/location definition
- **Integration Runtime (IR)**:
  - **Azure IR** for cloud-native movement
  - **Self-hosted IR** for on-prem / private network access
- **Triggers**:
  - schedule
  - tumbling window
  - event-based

Below is a production-style **Copy Activity** pipeline JSON from **ADLS Gen2 → Azure SQL Database**.

In [ ]:
adf_pipeline = {
    "name": "CopyTelemetryAlertsFromADLSToAzureSQL",
    "properties": {
        "activities": [
            {
                "name": "CopyAlertsParquetToSql",
                "type": "Copy",
                "policy": {
                    "timeout": "7.00:00:00",
                    "retry": 2,
                    "retryIntervalInSeconds": 30,
                    "secureOutput": False,
                    "secureInput": False,
                },
                "typeProperties": {
                    "source": {
                        "type": "ParquetSource",
                        "storeSettings": {
                            "type": "AzureBlobFSReadSettings",
                            "recursive": True,
                        },
                        "formatSettings": {
                            "type": "ParquetReadSettings"
                        },
                    },
                    "sink": {
                        "type": "AzureSqlSink",
                        "writeBehavior": "insert",
                        "sqlWriterUseTableLock": True,
                    },
                    "enableStaging": False,
                },
                "inputs": [{"referenceName": "ds_adls_alerts_parquet", "type": "DatasetReference"}],
                "outputs": [{"referenceName": "ds_sql_alerts_target", "type": "DatasetReference"}],
            }
        ],
        "annotations": [
            "source->sink",
            "integration-runtime",
            "schedule-trigger-ready",
        ],
    }
}

print(json.dumps(adf_pipeline, indent=2))

## 6) Event Hubs patterns

**Mental shortcut:** Event Hubs is **Kafka for Azure** — high-throughput ingestion, partitions, consumer groups, and Kafka protocol compatibility.

We will:

1. create an Event Hubs namespace  
2. create an event hub  
3. create **two consumer groups**  
4. send **20 telemetry events**  
5. read them back from each consumer group  
6. show that both groups independently receive the full stream

In [ ]:
def create_eventhub_namespace_and_hub():
    run_az(["account", "set", "--subscription", SUBSCRIPTION_ID], check=True)

    run_az([
        "eventhubs", "namespace", "create",
        "--name", eventhub_namespace,
        "--resource-group", resource_group,
        "--location", location,
        "--sku", "Standard",
    ], check=True)

    run_az([
        "eventhubs", "eventhub", "create",
        "--name", eventhub_name,
        "--resource-group", resource_group,
        "--namespace-name", eventhub_namespace,
        "--partition-count", "2",
        "--message-retention", "1",
    ], check=True)

    for cg in [consumer_group_a, consumer_group_b]:
        run_az([
            "eventhubs", "eventhub", "consumer-group", "create",
            "--consumer-group-name", cg,
            "--eventhub-name", eventhub_name,
            "--namespace-name", eventhub_namespace,
            "--resource-group", resource_group,
        ], check=True)

    cp = run_az([
        "eventhubs", "namespace", "authorization-rule", "keys", "list",
        "--resource-group", resource_group,
        "--namespace-name", eventhub_namespace,
        "--name", "RootManageSharedAccessKey",
        "--output", "json",
    ], check=True)
    keys = safe_json_loads(cp.stdout)
    return keys["primaryConnectionString"]

eventhub_connection_string = None
try:
    eventhub_connection_string = create_eventhub_namespace_and_hub()
    print("Event Hubs namespace and hub created.")
except Exception as exc:
    print("Event Hubs setup did not complete in this runtime.")
    print(type(exc).__name__, "->", exc)

In [ ]:
def build_events(count: int = 20) -> list[dict]:
    severities = ["INFO", "WARN", "ERROR", "CRITICAL"]
    regions = ["us-east", "us-west", "emea", "apac"]
    payloads = []
    for i in range(count):
        payloads.append({
            "sequence": i,
            "endpoint_id": 1000 + i,
            "region": regions[i % len(regions)],
            "latency_ms": round(25 + (i * 1.7), 2),
            "error_rate": round((i % 5) * 0.01, 3),
            "throughput_rps": 100 + i * 3,
            "severity": severities[i % len(severities)],
            "source": "citi-api-monitor",
        })
    return payloads

def send_events(connection_string: str, payloads: list[dict]) -> None:
    if EventHubProducerClient is None or EventData is None:
        raise RuntimeError("azure.eventhub is not available in this kernel.")
    producer = EventHubProducerClient.from_connection_string(
        conn_str=connection_string,
        eventhub_name=eventhub_name,
    )
    with producer:
        batch = producer.create_batch()
        for payload in payloads:
            evt = EventData(json.dumps(payload))
            try:
                batch.add(evt)
            except ValueError:
                producer.send_batch(batch)
                batch = producer.create_batch()
                batch.add(evt)
        if len(batch) > 0:
            producer.send_batch(batch)

telemetry_events = build_events(20)

try:
    if eventhub_connection_string:
        send_events(eventhub_connection_string, telemetry_events)
        print(f"Sent {len(telemetry_events)} telemetry events.")
    else:
        print("Skipping send because Event Hubs connection string is unavailable.")
except Exception as exc:
    print("Event send did not complete in this runtime.")
    print(type(exc).__name__, "->", exc)

In [ ]:
from collections import defaultdict

def read_events_from_consumer_group(connection_string: str, consumer_group: str, target_count: int = 20, timeout_seconds: int = 25):
    if EventHubConsumerClient is None:
        raise RuntimeError("azure.eventhub is not available in this kernel.")

    received = []
    seen_sequences = set()
    start_time = time.time()

    def on_event(partition_context, event):
        try:
            body = b"".join(b for b in event.body)
            payload = json.loads(body.decode("utf-8"))
        except Exception:
            payload = {"raw": str(event)}
        seq = payload.get("sequence", len(received))
        if seq not in seen_sequences:
            seen_sequences.add(seq)
            received.append(payload)
        partition_context.update_checkpoint(event)

    consumer = EventHubConsumerClient.from_connection_string(
        conn_str=connection_string,
        consumer_group=consumer_group,
        eventhub_name=eventhub_name,
    )

    try:
        with consumer:
            while time.time() - start_time < timeout_seconds and len(received) < target_count:
                consumer.receive(
                    on_event=on_event,
                    starting_position="-1",
                    max_wait_time=5,
                )
    finally:
        pass

    return received

consumer_results = {}

try:
    if eventhub_connection_string:
        consumer_results[consumer_group_a] = read_events_from_consumer_group(
            eventhub_connection_string, consumer_group_a, target_count=20, timeout_seconds=20
        )
        consumer_results[consumer_group_b] = read_events_from_consumer_group(
            eventhub_connection_string, consumer_group_b, target_count=20, timeout_seconds=20
        )

        summary = {k: len(v) for k, v in consumer_results.items()}
        print("Consumer group event counts:", summary)

        if all(len(v) == 20 for v in consumer_results.values()):
            print("Consumer group isolation confirmed: both groups received all 20 events.")
        else:
            print("One or more consumer groups did not receive the full expected set in this runtime.")
    else:
        print("Skipping read because Event Hubs connection string is unavailable.")
except Exception as exc:
    print("Event read did not complete in this runtime.")
    print(type(exc).__name__, "->", exc)

## 7) Cleanup

This deletes the resource group, which removes:

- storage account / ADLS Gen2 filesystem
- Event Hubs namespace + event hub
- any child resources created in the demo

In [ ]:
def cleanup_resource_group():
    run_az([
        "group", "delete",
        "--name", resource_group,
        "--yes",
        "--no-wait",
    ], check=True)

try:
    # Set to True when you want live Azure resources deleted at the end of the demo.
    PERFORM_CLEANUP = False
    if PERFORM_CLEANUP:
        cleanup_resource_group()
        print(f"Cleanup requested. Resource group delete submitted: {resource_group}")
    else:
        print(f"Cleanup skipped. To delete Azure resources, set PERFORM_CLEANUP=True for resource group: {resource_group}")
except Exception as exc:
    print("Cleanup command did not complete in this runtime.")
    print(type(exc).__name__, "->", exc)

## 8) What Just Happened

- **Synapse Serverless SQL** is Athena for Azure — pay per scan, no infrastructure.
- **ADF** is Glue for Azure — managed ETL with 90+ connectors.
- **Event Hubs** is Kafka for Azure — Kafka protocol compatible.
- This maps well to a Citi-style telemetry platform where thousands of endpoints generate operational metrics and alerts that must be landed, queried, and streamed reliably.